<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 13. GAN — Generative Adversarial Nets

- **원 논문:** [Generative Adversarial Nets](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html)
- **저자 / 발표:** Ian J. Goodfellow et al. · NeurIPS (2014)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** PyTorch 기본, binary cross entropy, 확률분포, 역전파

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

2차원 Gaussian 데이터에 대해 MLP 생성자와 판별자를 번갈아 갱신하고, 논문이 제안한 non-saturating 생성자 목적함수로 고정 noise 표본의 분포 오차가 줄어드는지 확인한다.

**원 논문과 다른 것**

논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3, Eq. (1) | `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수 | 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사 |
| §3, Algorithm 1 및 Eq. (1) 직후 문단 | `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신 | 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사 |
| §4.1, Proposition 1 Eq. (2), Theorem 1 및 Eq. (6) | `rbf_mmd`와 판별 확률로 생성 분포 수렴을 진단 | 학습 전후 MMD와 판별자 평균 출력을 비교 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\min_G\max_D\;
\mathbb{E}_{x\sim p_{data}}[\log D(x)]
+\mathbb{E}_{z\sim p(z)}[\log(1-D(G(z)))]
$$

- generator $G$는 noise를 sample로, discriminator $D$는 sample을 real 확률로 변환합니다.
- 두 모델의 목표가 반대인 minimax game이므로 optimizer와 gradient 경로를 분리합니다.
- discriminator step과 generator step을 독립 함수로 구현하고 해당 step의 parameter만 변하는지 보장합니다.
- $z:[B,D_z]\rightarrow G(z):[B,D_x]\rightarrow D(\cdot):[B,1]$의 흐름입니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3, Eq. (1)
- **노트북 구현:** `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문의 게임을 2차원으로 축소하기

논문 §3 Eq. (1)은

$$\min_G\max_D\;\mathbb E_{x\sim p_{data}}\log D(x)
+\mathbb E_{z\sim p_z}\log(1-D(G(z)))$$

를 정의합니다. 여기서는 타원형 2차원 Gaussian을 실제 분포로 두고 두 MLP를
번갈아 갱신합니다. 생성자에는 같은 절에서 권한 `maximize log D(G(z))`, 즉
non-saturating loss를 사용합니다. 이 변형은 고정점은 같고 초기 gradient가 더 강합니다.

**성공 기준:** 고정 noise에서 생성 분포와 실제 분포 사이 RBF-MMD가 초기보다 작아야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(130)
np.random.seed(130)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# This tiny teaching batch may be faster with AI_LAB_DEVICE=cpu;
# auto acceleration preserves the same code path for larger variants.

real_mean = torch.tensor([1.4, -0.8])
real_transform = torch.tensor([[0.75, 0.0], [0.28, 0.42]])


def sample_real(n, generator):
    base = torch.randn(n, 2, generator=generator)
    return (real_mean + base @ real_transform.T).to(DEVICE)


def sample_noise(n, generator):
    return torch.randn(n, 2, generator=generator).to(DEVICE)


data_rng = torch.Generator().manual_seed(131)
real_data = sample_real(1024, data_rng)
print(ACCELERATOR.summary())
print(f"real mean={real_data.mean(0).tolist()}, shape={tuple(real_data.shape)}")
assert real_data.shape == (1024, 2)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3, Eq. (1)
- **이 셀의 책임:** `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 2),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def discriminator_loss(real_logits, fake_logits):
    real_term = F.binary_cross_entropy_with_logits(
        real_logits,
        torch.ones_like(real_logits),
    )
    fake_term = F.binary_cross_entropy_with_logits(
        fake_logits,
        torch.zeros_like(fake_logits),
    )
    return real_term + fake_term


def generator_loss(fake_logits):
    # §3: minimax와 고정점이 같지만 초기 포화를 줄이는 대안.
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


G, D = ACCELERATOR.move(Generator(), Discriminator())
assert G(torch.zeros(5, 2, device=DEVICE)).shape == (5, 2)
assert D(torch.zeros(5, 2, device=DEVICE)).shape == (5,)
zero_logits = torch.zeros(2, device=DEVICE)
assert float(discriminator_loss(zero_logits, zero_logits)) > 0.0

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3, Algorithm 1 및 Eq. (1) 직후 문단
- **이 셀의 책임:** `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def gan_step(G, D, g_optimizer, d_optimizer, batch_size, generator):
    # Algorithm 1: k=1 discriminator update.
    real_batch = sample_real(batch_size, generator)
    noise = sample_noise(batch_size, generator)
    fake_batch = G(noise)
    d_optimizer.zero_grad()
    d_loss = discriminator_loss(D(real_batch), D(fake_batch.detach()))
    d_loss.backward()
    d_optimizer.step()

    # §3의 non-saturating 대안으로 generator update.
    noise = sample_noise(batch_size, generator)
    g_optimizer.zero_grad()
    for parameter in D.parameters():
        parameter.requires_grad_(False)
    g_loss = generator_loss(D(G(noise)))
    g_loss.backward()
    g_optimizer.step()
    for parameter in D.parameters():
        parameter.requires_grad_(True)
    return float(d_loss.detach()), float(g_loss.detach())


# detach가 없으면 이 검사 시 D update가 G에 불필요한 gradient를 남길 수 있다.

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3, Algorithm 1 및 Eq. (1) 직후 문단
- **이 셀의 책임:** `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
train_rng = torch.Generator().manual_seed(132)
probe_rng = torch.Generator().manual_seed(133)
probe_noise = sample_noise(384, probe_rng)
with torch.no_grad():
    initial_fake = G(probe_noise).clone()

g_optimizer = torch.optim.Adam(G.parameters(), lr=1.5e-3, betas=(0.5, 0.9))
d_optimizer = torch.optim.Adam(D.parameters(), lr=1.0e-3, betas=(0.5, 0.9))
history = []
for step in range(600):
    history.append(gan_step(G, D, g_optimizer, d_optimizer, 128, train_rng))

assert len(history) == 600
assert np.isfinite(np.asarray(history)).all()
print(f"final D loss={history[-1][0]:.3f}, G loss={history[-1][1]:.3f}")

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4.1, Proposition 1 Eq. (2), Theorem 1 및 Eq. (6)
- **이 셀의 책임:** `rbf_mmd`와 판별 확률로 생성 분포 수렴을 진단
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 학습 전후 MMD와 판별자 평균 출력을 비교. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def rbf_mmd(x, y, sigma=1.0):
    def kernel(a, b):
        squared_distance = torch.cdist(a, b).pow(2)
        return torch.exp(-squared_distance / (2.0 * sigma**2))

    return kernel(x, x).mean() + kernel(y, y).mean() - 2.0 * kernel(x, y).mean()


eval_rng = torch.Generator().manual_seed(134)
real_eval = sample_real(len(probe_noise), eval_rng)
with torch.no_grad():
    final_fake = G(probe_noise)
    initial_mmd = float(rbf_mmd(initial_fake, real_eval))
    final_mmd = float(rbf_mmd(final_fake, real_eval))
    mean_error = float((final_fake.mean(0) - real_eval.mean(0)).norm())
    d_real = float(torch.sigmoid(D(real_eval)).mean())
    d_fake = float(torch.sigmoid(D(final_fake)).mean())

print(f"RBF-MMD: {initial_mmd:.4f} -> {final_mmd:.4f}")
print(f"mean error={mean_error:.3f}, D(real)={d_real:.3f}, D(fake)={d_fake:.3f}")
assert math.isfinite(final_mmd) and final_mmd < initial_mmd
assert mean_error < 0.45

losses = np.asarray(history)
real_eval_cpu = real_eval.detach().cpu()
initial_fake_cpu = initial_fake.detach().cpu()
final_fake_cpu = final_fake.detach().cpu()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].scatter(
    real_eval_cpu[:, 0],
    real_eval_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="real",
)
axes[0].scatter(
    initial_fake_cpu[:, 0],
    initial_fake_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="initial G",
)
axes[0].set_title("before training")
axes[0].legend()
axes[1].scatter(
    real_eval_cpu[:, 0],
    real_eval_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="real",
)
axes[1].scatter(
    final_fake_cpu[:, 0],
    final_fake_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="final G",
)
axes[1].set_title("after adversarial training")
axes[1].legend()
axes[2].plot(losses[:, 0], alpha=0.75, label="D loss")
axes[2].plot(losses[:, 1], alpha=0.75, label="G loss")
axes[2].set(title="Algorithm 1 updates", xlabel="step")
axes[2].legend()
fig.tight_layout()
plt.show()

## 7. 해석 예시 답안

1. 두 목적함수가 함께 변하는 게임이므로 한 모델이 좋아지면 상대 모델의 loss가
   다시 커질 수 있습니다. 따라서 표본과 분포 metric도 함께 봐야 합니다.
2. 판별자 단계에서 D의 loss가 G까지 역전파되어 G를 갱신하는 경로를 끊습니다. G는 별도의 생성자 단계에서만 갱신됩니다.
3. $p_g=p_{data}$이면 최적 판별자 $D^*(x)=1/2$이므로 두 평균 모두 0.5에
   가까워집니다. 유한 MLP·유한 step에서는 정확히 0.5일 필요는 없습니다.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.